## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [14]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
# chroma - in memory vector store
from langchain_chroma import Chroma
# how many types of messages are there? is it arbitary?
from langchain_core.messages import SystemMessage, HumanMessage
# embedding model
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [16]:
MODEL = "gpt-4.1"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [18]:
# embedding model - we will use this to convert text into vectors and store them in the vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# vector store - we will use this to store the vectors and retrieve them later.
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [19]:
# retriever - lets as retrieve the relevant documents from the vector store based on the query. 
# store -> retreiver
retriever = vectorstore.as_retriever()
# llm - use this to generate the response based on the retrieved documents
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [20]:
docs = retriever.invoke("Who is Avery?")
docs

[Document(id='f793bfec-f1a7-4692-9fa5-b79f1202636c', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [21]:
# ask llm a q (without context)
llm.invoke("Who is Avery?")

AIMessage(content='The name "Avery" can refer to many different people, characters, or entities, depending on the context. Here are a few possibilities:\n\n1. **Given Name/Surname:** Avery is a common given name and surname for both males and females.\n\n2. **Fictional Characters:** There are several fictional characters named Avery in books, movies, and TV shows. For example, Avery Jessup from the TV show *30 Rock*, or Avery Barkley from *Nashville*.\n\n3. **Companies:** Avery Dennison is a well-known company specializing in labeling and packaging materials.\n\n4. **Places:** There are towns and counties named Avery in the United States.\n\nIf you have a specific context in mind (such as a book, TV show, or field), please provide more details so I can give a more precise answer!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 170, 'prompt_tokens': 11, 'total_tokens': 181, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'aud

## Time to put this together!

In [23]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [24]:
def answer_question(question: str, history):
    # get relevant docs and add to context
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    # join system prompt with context
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    # with (system prompt + context) + question, ask the llm to generate a response
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [25]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She co-founded the company in 2015 and has played a key role in guiding Insurellm to become a leading Insurance Tech provider. Avery is recognized for her innovative leadership strategies and expertise in risk management, which have helped propel the company into the mainstream insurance market. She is based in San Francisco, California.'

## What could possibly come next? 😂

In [26]:
gr.ChatInterface(answer_question).launch()

/Users/jay/repos/study/udemy/llm-ed/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


* history is not being passed
* and rag look up is happening on each new question
* the fix is to pass history and to look up the context against each of the question

## Admit it - you thought RAG would be more complicated than that!!